In [ ]:
# Upload the codebase into the input folder, this ensures that
# the Python interpreter can load python files from within it.

import sys
sys.path.append('/kaggle/input/datasets/joelbehzhirong/uowaisecurity-codebase')  # change the parameter to your path if it's different
sys.path.append('/kaggle/input/datasets/joelbehzhirong/uowaisecurity-codebase/codebase')  # change the parameter to your path if it's different

In [ ]:
import torch
import sys

device = torch.device('cuda' if torch.cuda.is_available else 'cpu')

print('PyTorch Version:', torch.__version__)
print('-' * 60)
if torch.cuda.is_available():
    print('CUDA Device Count:', torch.cuda.device_count())
    print('CUDA Device Name:')
    for i in range(torch.cuda.device_count()):
        print('\t', torch.cuda.get_device_name(i))
    print('CUDA Current Device Index:', torch.cuda.current_device())
    print('-' * 60)

print(f"Python version = {sys.version}")


In [ ]:
# As usual, a bit of setup
import matplotlib.pyplot as plt
import types
from pathlib import Path
import logging
logging.getLogger().setLevel(logging.INFO)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

exp_cfg = types.SimpleNamespace()
exp_cfg.data_dir = Path(f"/kaggle/working/data")
exp_cfg.out_dir = Path(f"/kaggle/working/out")

exp_cfg.data_dir.mkdir(parents=True, exist_ok=True)
exp_cfg.out_dir.mkdir(parents=True, exist_ok=True)

exp_cfg.device = torch.device('cuda:0')  # use the first GPU


# Task 1/2: Grey-box Adversarial Examples (Total: 8 marks)

You will implement grey-box adversarial examples to fool a pretrained target model. Please download the target model from https://uowmailedu-my.sharepoint.com/:u:/g/personal/wzong_uow_edu_au/EWHR470-0khEs2gqOdAAZOMBHI7_KWubdkiJCD4pNpiCRQ?e=tra4CC

Unzip the file and upload the "target_model" folder to the input folder.
The following code will load the target model and evaluate its performance.

In this task, you have partial knowlegde about this target:
- The architecture of the target model is VGG (vgg11_bn).
- This model was trained in exactly the same way as we trained VGG models for the CW attack lab task.

Given 100 images that can be correctly recognized by the target model, you need to generate **targeted** adversarial examples for each image.
Adversarial perturbations are constrained by $L_{∞} \le 0.04$.

To generate adversarial examples, you need to implement the function **generate_greybox_ae** in the corresponding cell below.
Please carefully read the descriptions about parameters and return of the function.
You cannot use the target model, or its copy, to calculate any gradients.


Marks:
- 6 marks if fooling rate > 80%.
- 4 marks if fooling rate > 60%.
- 2 marks if fooling rate > 40%.
- 1 mark if fooling rate > 20%.
- 0 marks otherwise.

## Hints

One potential solution is to follow the successful targeted black-box attack strategy we have discussed in the lecture. By including the knowledge about the target model (e.g., its architecture, training data, etc.), you can generate targeted grey-box attacks.

You can try both CW and PGD attacks, and choose the one that performs well for this task. You may need to explore effective attack parameters for achieving full marks.

You can load a copy of the target model in your function and access its predictions, i.e., observing pairs of input and output. although this is not necessary for implementing grey-box attacks.

However, you are **NOT** allowed to use the target model, or its copy, to calculate any gradients. Such behaviors will result in **0 marks** for the coding part.


## Loading images
Run the cell below to load 100 clean images that can be correctly recognized by the target model. These images will be saved in x_arr in the unnormalized form.

In [ ]:

#My is named "saved_epoch-49.pth.tar" as i had trouble uploading the model.
#This is a workaround 

import shutil

target_ckpt_path = Path("/kaggle/input/models/joelbehzhirong/target-model/pytorch/default/1/target_model/saved_epoch-49.pth.tar")
renamed_path = Path("/kaggle/working/saved_epoch-49.tar")
shutil.copy(target_ckpt_path, renamed_path)

target_ckpt_dir = Path("/kaggle/working")

In [ ]:

import numpy as np
import torchvision
import torchvision.transforms as transforms
from codebase import model_trainer, utils, setup
from codebase.classifiers import vgg

# input to this model must be normalized
cifar10_mean_tensor = torch.Tensor(setup.CIFAR10_MEAN).reshape([1, 3, 1, 1]).to(exp_cfg.device)
cifar10_std_tensor = torch.Tensor(setup.CIFAR10_STD).reshape([1, 3, 1, 1]).to(exp_cfg.device)

# IMPORTANT! ------
# Im using load_ckpt() directly instead of load_latest_ckpt(),
# This is because load_latest_ckpt() fails to parse the epoch number from "saved_epoch-49.pth.tar",
# a double file extension causes Path.stem to leave "49.pth", which int() can't parse.
#---------------------
target_ckpt_path = Path("/kaggle/input/models/joelbehzhirong/target-model/pytorch/default/1/target_model/saved_epoch-49.pth.tar")
dic_saved = model_trainer.ModelTrainer.load_ckpt(target_ckpt_path)
assert dic_saved is not None, "Cannot find pretrained weights."

target_model = vgg.vgg11_bn(num_classes=10).to(exp_cfg.device)
target_model.load_state_dict(dic_saved["model_state"])
target_model.eval()

# evaluate it on the normalized testing set.

test_set = torchvision.datasets.CIFAR10(
        root=str(exp_cfg.data_dir), train=False,
        download=True, transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(setup.CIFAR10_MEAN, setup.CIFAR10_STD)
        ])
)

model_trainer.ModelTrainer.eval_on_dset(target_model, test_set)

# create a test set that is NOT normalized for extracting clean images
unnormalized_test_set = torchvision.datasets.CIFAR10(
    root=str(exp_cfg.data_dir), train=False,
    download=True, transform=transforms.Compose([transforms.ToTensor()])
)

# randomly get 100 images that are correctly recognized by the target model
rand_idxes = np.random.RandomState(375).permutation(len(unnormalized_test_set))
x_arr, y_arr = [], []

for idx in rand_idxes:
    test_x, test_y = unnormalized_test_set[idx]
    test_x = test_x.to(exp_cfg.device)
    assert isinstance(test_y, int)

    pred = target_model(utils.normalize(test_x.unsqueeze(0), cifar10_mean_tensor, cifar10_std_tensor)).argmax(dim=1)
    if pred.item() == test_y:
        x_arr.append(test_x)
        y_arr.append(test_y)

    if len(x_arr) >= 100:
        break
assert len(x_arr) == 100, "Cannot find enough correctly predicted clean images."

x_arr = torch.stack(x_arr).to(exp_cfg.device)
y_arr = torch.LongTensor(y_arr).to(exp_cfg.device)

# Before generating any attacks, let's check the predictions on x_arr.
preds = target_model(utils.normalize(x_arr, cifar10_mean_tensor, cifar10_std_tensor)).argmax(dim=1)
acc = (preds == y_arr).sum() / len(y_arr)
print(f"Accuracy of the target model on x_arr = {acc * 100:.1f}%")

# show some clean images
utils.show_imgs_tensor(nrows=2, ncols=5, imgs_arr=x_arr, labels_arr=y_arr, class_names=setup.CIFAR10_CLASSES)

## Implement generate_greybox_ae (6 marks)
Implement your function in the cell below. Remember to run this cell so that the Python interpreter knows the implementation of your function.

In [ ]:

import torch
import torch.nn.functional as F

# ===== Train surrogate models =====
train_set_norm = torchvision.datasets.CIFAR10(
    root=str(exp_cfg.data_dir), train=True, download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(setup.CIFAR10_MEAN, setup.CIFAR10_STD)
    ])
)
test_set_norm = torchvision.datasets.CIFAR10(
    root=str(exp_cfg.data_dir), train=False, download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize(setup.CIFAR10_MEAN, setup.CIFAR10_STD)])
)

surrogates = []
for seed in [0, 1, 2, 3]:
    torch.manual_seed(seed)
    surrogate = vgg.vgg11_bn(num_classes=10).to(exp_cfg.device)
    cfg = model_trainer.ModelTrainerConfig(
        device=exp_cfg.device, max_epochs=50, batch_size=128,
        ckpt_dir=exp_cfg.out_dir / f"surrogate_vgg_aug_{seed}",
        save_every_epoch=50,
    )
    trainer = model_trainer.ModelTrainer(surrogate, train_set_norm, test_set_norm, cfg)
    trainer.train()
    surrogate.eval()
    surrogates.append(surrogate)

print(f"Trained {len(surrogates)} surrogates")
for i, s in enumerate(surrogates):
    print(f"Surrogate {i}:")
    model_trainer.ModelTrainer.eval_on_dset(s, test_set_norm)


# ===== Attack functions =====
def margin_loss_targeted(logits, target_labels, kappa=0.0):
    one_hot = F.one_hot(target_labels, logits.size(1)).float()
    target_logit = (logits * one_hot).sum(dim=1)
    other_logit = (logits - one_hot * 1e4).max(dim=1)[0]
    loss = torch.clamp(other_logit - target_logit + kappa, min=0)
    return loss.mean()

def input_diversity(x, resize_prob=0.5, low=29, high=33):
    if torch.rand(1).item() > resize_prob:
        return x
    rnd = torch.randint(low, high, (1,)).item()
    x_resized = F.interpolate(x, size=(rnd, rnd), mode='nearest')
    pad_total = high - rnd
    pad_left = torch.randint(0, pad_total + 1, (1,)).item()
    pad_top = torch.randint(0, pad_total + 1, (1,)).item()
    x_padded = F.pad(x_resized, [pad_left, pad_total - pad_left, pad_top, pad_total - pad_top])
    return F.interpolate(x_padded, size=x.shape[-2:], mode='nearest')

def generate_greybox_ae(
        x_arr: torch.Tensor,
        y_arr: torch.Tensor,
        adv_target_arr: torch.Tensor,
        adv_l_inf: float,
        exp_cfg,
) -> torch.Tensor:
    """
    x_arr: unnormalized clean images with shape [N, C, H, W].
    y_arr: ground truth labels with shape [N].
    adv_target_arr: the target labels of adversarial examples with shape [N].
    adv_l_inf: the allowed L_inf for all adversarial examples.
    exp_cfg: general configurations including out_dir, data_dir, device, etc.
    return: adversarial examples with shape [N, C, H, W] that can fool the target model.
    """
    alpha = 0.0012
    steps = 900
    decay = 1.0
    use_di = True
    di_samples = 3
    scales = [1.0, 0.75]
    kappa = 12.0

    device = exp_cfg.device
    images = x_arr.clone().detach().to(device)
    target_labels = adv_target_arr.clone().detach().to(device)

    rand_init = (torch.rand_like(images) * 2 - 1) * adv_l_inf
    adv = torch.clamp(images + rand_init, 0.0, 1.0).detach()

    momentum = torch.zeros_like(images)

    for step in range(steps):
        if step % 100 == 0:
            print(f"  attack step {step}/{steps}")

        adv.requires_grad_(True)
        total_loss = 0.0
        n_terms = 0
        for scale in scales:
            scaled_adv = adv * scale
            for _ in range(di_samples if use_di else 1):
                for model in surrogates:
                    inp = input_diversity(scaled_adv) if use_di else scaled_adv
                    inp_norm = utils.normalize(inp, cifar10_mean_tensor, cifar10_std_tensor)
                    logits = model(inp_norm)
                    total_loss = total_loss + margin_loss_targeted(logits, target_labels, kappa=kappa)
                    n_terms += 1
        total_loss = total_loss / n_terms

        grad = torch.autograd.grad(total_loss, adv)[0]
        grad_norm = grad / (grad.abs().mean(dim=(1, 2, 3), keepdim=True) + 1e-12)
        momentum = decay * momentum + grad_norm

        with torch.no_grad():
            adv = adv - alpha * momentum.sign()
            delta = torch.clamp(adv - images, min=-adv_l_inf, max=adv_l_inf)
            adv = torch.clamp(images + delta, 0.0, 1.0)
        adv = adv.detach()

    return adv

print("Attack functions ready (steps=900)")

The cell below will evaluates your implementation and print out your marks for this part.

In [ ]:
# generate grey-box adversarial examples using your function
adv_l_inf = 0.04
# choose target label as the next label.
adv_target_arr = (y_arr + 1) % len(setup.CIFAR10_CLASSES)

adv_x = generate_greybox_ae(
    x_arr=x_arr, y_arr=y_arr, adv_target_arr=adv_target_arr, adv_l_inf=adv_l_inf, exp_cfg=exp_cfg
)
assert target_model.training is False, "Your function should not change the training state."

# check the quality of adv_x
cal_adv_l_inf = torch.abs(adv_x - x_arr).max().item()
print(f"Maximum l_inf = {cal_adv_l_inf:.2f}")
if (cal_adv_l_inf > adv_l_inf + 1e-4) or (adv_x.max().item() > 1.0) or (adv_x.min().item() < 0.0):
    print("At least one adversarial example does not fulfill the requirement.")
    adv_marks = 0
else:
    # get marks if achieving ceratin fooling rate
    with torch.no_grad():
        adv_preds = target_model(utils.normalize(adv_x, cifar10_mean_tensor, cifar10_std_tensor)).argmax(dim=1)
    fooling_rate = (adv_preds == adv_target_arr).sum() / len(adv_target_arr)
    fooling_rate = fooling_rate.item()
    print(f"Grey-box adversarial examples fooling rate = {fooling_rate * 100:.1f}%")

    if fooling_rate > 0.80:
        adv_marks = 6
    elif fooling_rate > 0.60:
        adv_marks = 4
    elif fooling_rate > 0.40:
        adv_marks = 2
    elif fooling_rate > 0.20:
        adv_marks = 1
    else:
        adv_marks = 0

    utils.show_imgs_tensor(nrows=2, ncols=5, imgs_arr=adv_x, labels_arr=adv_preds, class_names=setup.CIFAR10_CLASSES)
print(f"\n******* Your marks for generating grey-box adversarial examples is {adv_marks}. *******\n")

## Briefly describe how you implemented the grey-box adversarial examples. (2 marks)

Your answer: 
I trained several vgg11_bn models which has the same architecture as the target with different random seeds, and used data augmentation so they'd generalize similarly to the real target rather than just memorizing training data. Using multiple surrogates meant the attack had to work against several slightly different opinions at once, which made it less likely to overfit to one model's quirks.

For the attack, i used an iterative approach. This works by nudging the image's pixels a tiny bit at a time while staying in the allowed pixel change budget. 
Step size: Started with alpha=0.004, steps=60 but this overshot so changed to alpha=0.0012-0.0015, steps=600-900.
Momentum: decay=1.0, accumulating gradient direction across steps for consistency
Input diversity: randomly resized images to 29-33px with resize_prob=0.5, averaging over di_samples=3 augmented views per step.
Scale invariance: average gradients across scales=[1.0, 0.75]
Margin loss confidence (kappa): raised from 0 to 8 to 12, forcing more confident win margin against surrogates.
Random start: began from a random offset within the +-0.04 eps ball rather than the clean image itself.

# Task 2/2: Universal Adversarial Perturbations (Total: 7 marks)

You will implement universal adversarial perturbations (UAPs) to fool a pretrained target model.
You will use the same target model from Task 1/2 to complete this task.
This is a white-box attack so that you have full access to the target model.

You are given 100 images that can be correctly recognized by the target model.
You need to generate UAPs that can be added to all these 100 images to fool the target model to output 3 (i.e., "cat").
UAPs are constrained by $L_{∞} \le 0.06$.

To generate UAPs, you need to implement the function **generate_UAPs** in the corresponding cell below.
Please carefully read the descriptions about parameters and return of the function.

Marks:
- 5 marks if fooling rate > 90%.
- 3 marks if fooling rate > 70%.
- 1 mark if fooling rate > 50%.
- 0 marks otherwise.

## Implement generate_UAPs (5 marks)
Implement your function in the cell below. Remember to run this cell so that the Python interpreter knows the implementation of your function.

In [ ]:
def generate_UAPs(
        target_model: torch.nn.Module,
        x_arr: torch.Tensor,
        y_arr: torch.Tensor,
        UAP_target: int,
        UAP_l_inf: float,
        exp_cfg,
) -> torch.Tensor:
    """
    target_model: the target model we want to fool.
    x_arr: unnormalized clean images with shape [N, C, H, W].
    y_arr: ground truth labels with shape [N].
    UAP_target: the target label.
    UAP_l_inf: the allowed l_inf for the UAPs.
    exp_cfg: general configurations including out_dir, data_dir, device, etc.
    return: UAPs with shape [C, H, W] that can fool the target model when added to any image.
    """
    alpha = 0.006
    steps = 1000
    decay = 1.0
    kappa = 5.0

    device = exp_cfg.device
    images = x_arr.clone().detach().to(device)
    N, C, H, W = images.shape
    target_labels = torch.full((N,), UAP_target, dtype=torch.long, device=device)

    delta = (torch.rand(1, C, H, W, device=device) * 2 - 1) * UAP_l_inf
    delta = delta.detach()

    momentum = torch.zeros_like(delta)

    target_model.eval()

    for step in range(steps):
        delta.requires_grad_(True)

        adv = torch.clamp(images + delta, 0.0, 1.0)
        adv_norm = utils.normalize(adv, cifar10_mean_tensor, cifar10_std_tensor)
        logits = target_model(adv_norm)

        one_hot = F.one_hot(target_labels, logits.size(1)).float()
        target_logit = (logits * one_hot).sum(dim=1)
        other_logit = (logits - one_hot * 1e4).max(dim=1)[0]
        loss = torch.clamp(other_logit - target_logit + kappa, min=0).mean()

        grad = torch.autograd.grad(loss, delta)[0]
        grad_norm = grad / (grad.abs().mean(dim=(1, 2, 3), keepdim=True) + 1e-12)
        momentum = decay * momentum + grad_norm

        with torch.no_grad():
            delta = delta - alpha * momentum.sign()
            delta = torch.clamp(delta, min=-UAP_l_inf, max=UAP_l_inf)
        delta = delta.detach()

        if step % 100 == 0:
            with torch.no_grad():
                preds = logits.argmax(dim=1)
                fooling = (preds == target_labels).float().mean().item()
            print(f"  step {step}/{steps}, fooling rate so far = {fooling*100:.1f}%")

    return delta.squeeze(0)

The cell below will evaluate your implementation and print out your marks for this part.

In [ ]:
# generate UAPs with the target label = 3 (cat)
UAP_l_inf = 0.06
UAP_target = 3

UAPs = generate_UAPs(
    target_model=target_model, x_arr=x_arr, y_arr=y_arr,
    UAP_target=UAP_target, UAP_l_inf=UAP_l_inf, exp_cfg=exp_cfg
)
UAPs = UAPs.unsqueeze(0)
assert target_model.training is False, "Your function should not change the training state."

# check the quality of UAPs
cal_UAP_l_inf = torch.abs(UAPs).max().item()
print(f"UAPs l_inf = {cal_UAP_l_inf:.2f}")
if cal_UAP_l_inf > UAP_l_inf + 1e-4:
    print("At least one adversarial example does not fulfill the requirement.")
    UAP_marks = 0
else:
    # get marks if achieving ceratin fooling rate
    UAP_attack = torch.clamp(x_arr + UAPs, min=0, max=1)
    with torch.no_grad():
        UAP_preds = target_model(utils.normalize(UAP_attack, cifar10_mean_tensor, cifar10_std_tensor)).argmax(dim=1)
    UAP_fooling_rate = (UAP_preds == UAP_target).sum() / len(UAP_preds)
    UAP_fooling_rate = UAP_fooling_rate.item()
    print(f"UAPs fooling rate = {UAP_fooling_rate * 100:.1f}%")

    if UAP_fooling_rate > 0.90:
        UAP_marks = 5
    elif UAP_fooling_rate > 0.70:
        UAP_marks = 3
    elif UAP_fooling_rate > 0.50:
        UAP_marks = 1
    else:
        UAP_marks = 0

    utils.show_imgs_tensor(nrows=2, ncols=5, imgs_arr=UAP_attack, labels_arr=UAP_preds, class_names=setup.CIFAR10_CLASSES)

print(f"\n *******Your marks for generating UAPs is {UAP_marks}. *******\n")

## Briefly describe how you implemented the UAPs. (2 marks)

Your answer: 
I optimised a single shared perturbation directly against the real target model.
Perturbation: Instead of one adversarial example per image, i optimised a single delta with shape 1,C,H,W, broadcast across all 100 images, so the loss reflected how well the same perturbation fools every image at once.
Loss function: Used a margin style loss where i compared the target class's logit against the best competing class's logit, with a confidence margin (kappa), averaged over the batch of 100 images.
Random start: Began delta at random point within the += 0.06 eps ball rather than zero.
Momentum: accumulated gradient direction across steps (decay 1.0) for smoother updates

Tuning process:
First attempt: alpha=0.005, steps=500, kappa=10.0. This achieved 90% fooling rate.
Second attempt: alpha=0.003, steps=1000, kappa=20.0. This achieved 88% fooling rate.
Third attempt: alpha=0.006, steps=1000, kappa=5.0. This achieved 93% fooling rate.

In [ ]:
# here is your final marks for the coding part
print(f"\n *******Your total marks for the coding part is {adv_marks + UAP_marks} / 11. *******\n")

# Optional Task: Adaptive Attack (Total: 2 bonus marks)

This is an optional task for CSIT375 students.
There are 2 bonus marks for this task.
These marks can be used to offset your lost marks in the previous tasks.
You are encouraged to complete this task to practice adaptive attacks.

You will implement adaptive attacks to bypass a defence.
You will use the same target model from Task 1.
However, this model is protected by a defence which crops random portions of an input image and resizes the portions back to the original size.
This is a white-box attack so that you have full access to the target model as well as the defence.

We only use 50 clean images to generate attack in this task. This saves computational time. You need to generate targeted adversarial examples for each image. Adversarial perturbations are constrained by $L_{∞}\le0.04$.

To generate adversarial examples, you need to implement the function **generate_adaptive_attack** in the corresponding cell below. Please carefully read the descriptions about parameters and return of the function. You have full access to the target model.

Marks:
- 1 mark if fooling rate > 90%.
- 0 marks otherwise.

Hint: the defence is stochastic. You can run your solution multiple times until you achieve the highest marks.

The defence strategy is implemented in **RobustModel** in the following cell.
You will find that the defence significantly lowers the accuracy of the target model. This is because the defence strategy was not included in data augmentation when training the target model. At the same time, you should also observe that this defence destroys the grex-box attacks you generated in Task 1.

We focus on adaptive attacks in this task. We will defeat this defence even though it introduces too much distortion for the target model. This demonstrates of power of adaptive attacks.

In [ ]:
import torch.nn.functional as F
import torch.nn as nn

class RobustModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

        # random cropping as the defense
        self.transform = transforms.RandomResizedCrop(size=32, scale=(0.20, 0.50), antialias=True)

        # number of votes
        self.num_votes = 21

    def forward(self, x):
        # transform x multiple times and then the final output is the most voted one
        rlt_arr = []
        for _ in range(self.num_votes):
            trans_x = self.transform(x)
            logits = self.model(trans_x)
            preds = torch.argmax(logits, dim=1)
            rlt_arr.append(preds)

        rlt_arr = torch.stack(rlt_arr).transpose(0, 1)

        # the mode function returns the element with the highest count
        mode_val, mode_idx = torch.mode(rlt_arr, dim=1)

        # creat logits with the most voted one equal to 1 and the other logit values equal to 0
        # this is equivalent to one-hot encoding
        # note that this operation is not differentiable
        final_logits = F.one_hot(mode_val, num_classes=10).float()
        return final_logits

assert target_model.training is False
robust_model = RobustModel(target_model).to(exp_cfg.device)
robust_model.eval()

# evaluate the performance on clean testing data.
_, robust_acc, _ = model_trainer.ModelTrainer.eval_on_dset(robust_model, test_set)


Run the following cell to evaluate this robust model against your previously generated adversarial examples.

You can skip this cell if you have not done Task 1 yet.

In [ ]:
# This robust model should destroy your previously generated adversarial examples.
with torch.no_grad():
    adv_preds = robust_model(utils.normalize(adv_x, cifar10_mean_tensor, cifar10_std_tensor)).argmax(dim=1)
fooling_rate = (adv_preds == adv_target_arr).sum() / len(adv_target_arr)
print(f"Robust model: Task 1 grey-box adversarial examples fooling rate = {fooling_rate * 100:.1f}%")

Visualize how RandomResizedCrop works.

In [ ]:
tmp_test_loader = torch.utils.data.DataLoader(test_set, batch_size=5, num_workers=0, shuffle=False)
for vis_x, vis_y in tmp_test_loader:
    vis_x = vis_x.to(exp_cfg.device)

    vis_arr = [vis_x]
    for _ in range(4):
        vis_trans_x = robust_model.transform(vis_x)
        vis_arr.append(vis_trans_x)

    vis_arr = torch.concatenate(vis_arr, dim=0)
    vis_arr = vis_arr * cifar10_std_tensor + cifar10_mean_tensor

    vis_y = torch.tile(vis_y, (5, ))
    utils.show_imgs_tensor(nrows=5, ncols=5, imgs_arr=vis_arr, labels_arr=vis_y,
                            class_names=setup.CIFAR10_CLASSES)

    # only plot 1 batch
    break

## Implement generate_adaptive_attack (1 bonus mark)
Implement your function in the cell below. Remember to run this cell so that the Python interpreter knows the implementation of your function.

In [ ]:
def generate_adaptive_attack(
        target_model: torch.nn.Module,
        x_arr: torch.Tensor,
        y_arr: torch.Tensor,
        adv_target_arr: torch.Tensor,
        adv_l_inf: float,
        exp_cfg,
) -> torch.Tensor:
    """
    target_model: the target model to attack
    x_arr: unnormalized clean images with shape [N, C, H, W].
    y_arr: ground truth labels with shape [N].
    adv_target_arr: the target labels of adversarial examples with shape [N].
    adv_l_inf: the allowed l_inf for all adversarial examples.
    exp_cfg: general configurations including out_dir, data_dir, device, etc.

    return: adversarial examples with shape [N, C, H, W] that can fool the target model.
    """
    alpha = 0.003
    steps = 500
    decay = 1.0
    kappa = 20.0
    eot_samples = 30

    device = exp_cfg.device
    images = x_arr.clone().detach().to(device)
    target_labels = adv_target_arr.clone().detach().to(device)

    crop_transform = transforms.RandomResizedCrop(size=32, scale=(0.20, 0.50), antialias=True)

    rand_init = (torch.rand_like(images) * 2 - 1) * adv_l_inf
    adv = torch.clamp(images + rand_init, 0.0, 1.0).detach()

    momentum = torch.zeros_like(images)

    target_model.eval()

    for step in range(steps):
        adv.requires_grad_(True)
        total_loss = 0.0

        for _ in range(eot_samples):
            cropped = crop_transform(adv)
            cropped_norm = utils.normalize(cropped, cifar10_mean_tensor, cifar10_std_tensor)
            logits = target_model(cropped_norm)

            one_hot = F.one_hot(target_labels, logits.size(1)).float()
            target_logit = (logits * one_hot).sum(dim=1)
            other_logit = (logits - one_hot * 1e4).max(dim=1)[0]
            loss = torch.clamp(other_logit - target_logit + kappa, min=0).mean()
            total_loss = total_loss + loss

        total_loss = total_loss / eot_samples

        grad = torch.autograd.grad(total_loss, adv)[0]
        grad_norm = grad / (grad.abs().mean(dim=(1, 2, 3), keepdim=True) + 1e-12)
        momentum = decay * momentum + grad_norm

        with torch.no_grad():
            adv = adv - alpha * momentum.sign()
            delta = torch.clamp(adv - images, min=-adv_l_inf, max=adv_l_inf)
            adv = torch.clamp(images + delta, 0.0, 1.0)
        adv = adv.detach()

        if step % 100 == 0:
            print(f"  step {step}/{steps}, batch loss = {total_loss.item():.4f}")

    return adv

The cell below will evaluate your implementations and print out your marks for this part.

In [ ]:
adaptive_num = 50
adaptive_adv_l_inf = 0.04
adaptive_adv_x = generate_adaptive_attack (
    target_model, x_arr=x_arr[:adaptive_num], y_arr=y_arr[:adaptive_num], adv_target_arr=adv_target_arr[:adaptive_num],
    adv_l_inf=adaptive_adv_l_inf, exp_cfg=exp_cfg
)
assert target_model.training is False, "Your function should not change the training state."

# check the quality of adaptive_adv_x
cal_adaptive_adv_l_inf = torch.abs(adaptive_adv_x - x_arr[:adaptive_num]).max().item()
print(f"Maximum l_inf = {cal_adaptive_adv_l_inf:.2f}")
if (cal_adaptive_adv_l_inf > adaptive_adv_l_inf + 1e-4) or (adaptive_adv_x.max().item() > 1.0) or (adaptive_adv_x.min().item() < 0.0):
    print("At least one adversarial example does not fulfill the requirement.")
    adaptive_adv_marks = 0
else:
    # get marks if achieving certain fooling rate
    with torch.no_grad():
        adaptive_adv_preds = robust_model(utils.normalize(adaptive_adv_x, cifar10_mean_tensor, cifar10_std_tensor)).argmax(dim=1)
    assert len(adaptive_adv_preds) == adaptive_num

    fooling_rate = (adaptive_adv_preds == adv_target_arr[:adaptive_num]).sum() / adaptive_num
    fooling_rate = fooling_rate.item()
    print(f"{adaptive_num} adaptive adversarial examples fooling rate = {fooling_rate * 100:.1f}%")

    if fooling_rate > 0.90:
        adaptive_adv_marks = 1
    else:
        adaptive_adv_marks = 0

    utils.show_imgs_tensor(nrows=2, ncols=5, imgs_arr=adaptive_adv_x, labels_arr=adaptive_adv_preds,
                            class_names=setup.CIFAR10_CLASSES)
print(f"\n******* Your marks for generating adaptive adversarial examples is {adaptive_adv_marks}. *******\n")

## Briefly describe how you implemented the adaptive attacks. (1 bonus mark)

Your answer:
Since i have access to target_model and the crop transform, i used EOT (expectation over Tranformation), attacking traget_model directly, but averaging the loss/gradient over many randomly-cropped views of the image each step, so the perturbation survives regardless of which random crop the defense happens to sample at evaluation time.

Same crop transforms as the defense: reused RandomResizedCrop(size=32, scale=(0.20, 0.50)), matching the defense exactly, so the attack was optimized against the same distribution of crops it would actually be evaluated on.
EOT averaging: On each step, sampled the crop transform eot_samples times, ran each cropped view through target_model, and averaged the margin loss across all samples before computing gradients.
Margin loss, momentum, random start: reused the same components from the above tasks.

Tuning process:
First attempt: eot_samples=15, steps=300, kappa=10.0. This had inconsistent results (92%,88%,88%)
Second attempt: Since the defense itself votes over 21 crops internally, i reasoned the attack should average over at least as many crops to be optimised against a similarly broad distribution. 
eot_samples=30,, steps=500, kappa=20.0. This produced better scores (92%,94%,96%)


# Submission

After running all cells in this notebook, click "File -> Download Notebook" to save this notebook as "assignment1-csit375.ipynb" locally.
Please open this file with Kaggle or another Python notebook editor (e.g., Colab) again to confirm that all the results are correctly shown.

For submission, you only need to submit **assignment1-csit375.ipynb**.

Zip this file into a single **zip** file (do NOT use .rar). Submit this zip file via Moodle by the due date and time. Assignments that are not submitted on Moodle will not be marked.